# NimbusPay — fresh start (v2)
Use a **T4 GPU**. Stop old training first. This notebook uses `MyDrive/NimbusPayAssessment_Fresh_v2`; it never deletes or reuses old experiment results.

Run cells in order. Stay for Drive/upload prompts and the GPU test, then let the exploration cell run. It prints live progress and saves logs/checkpoints to Drive. Free Colab availability is not guaranteed. Exploration includes all five experiments and may take several hours. The later reproduction run is a separate fresh-runtime check.

Upload **NimbusPay_Fresh_Project.zip** and the original **training_assessment.zip** when requested. Do not upload the old project ZIP.

In [1]:
import time, sys, os, json, subprocess, shutil, zipfile, io
from pathlib import Path
STARTED=time.perf_counter()
from google.colab import drive, files
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/NimbusPayAssessment_Fresh_v2')
ROOT.mkdir(parents=True,exist_ok=True)
PROJECT=ROOT/'project'
PACK=ROOT/'candidate_pack'
print('Fresh workspace:',ROOT)


Mounted at /content/drive
Fresh workspace: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2


In [2]:
if not (PROJECT/'pipeline.py').exists() or not (PACK/'data/train.jsonl').exists():
    print('Select BOTH NimbusPay_Fresh_Project.zip and training_assessment.zip')
    uploaded=files.upload()
    for name, data in uploaded.items():
        if not name.endswith('.zip'): continue
        target=Path('/content/nimbuspay_v2_upload')/Path(name).stem
        target.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            for member in z.infolist():
                assert (target/member.filename).resolve().is_relative_to(target.resolve())
            z.extractall(target)
        if name.startswith('NimbusPay_Fresh_Project'):
            for source in target.rglob('pipeline.py'):
                shutil.copytree(source.parent,PROJECT,dirs_exist_ok=True)
        for source in target.rglob('ASSIGNMENT.md'):
            shutil.copytree(source.parent,PACK,dirs_exist_ok=True)
    del uploaded
assert (PROJECT/'pipeline.py').exists(), 'New project ZIP missing'
assert (PACK/'data/train.jsonl').exists(), 'Original assignment ZIP missing'
os.chdir(PROJECT)


## Install and test before training
This installs the CUDA-13-capable bitsandbytes pin and tests actual NF4 quantization. Dependencies are imported in fresh child processes, so this notebook kernel does not need a restart.

In [3]:
def run_live(command, log_path):
    log_path=Path(log_path)
    log_path.parent.mkdir(parents=True,exist_ok=True)
    print('Log:',log_path,flush=True)
    with log_path.open('a') as log:
        with subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
            try:
                for line in proc.stdout:
                    print(line,end='',flush=True)
                    log.write(line);log.flush()
                rc=proc.wait()
            except KeyboardInterrupt:
                proc.terminate();proc.wait()
                raise
    if rc: raise RuntimeError(f'Command failed ({rc}). Read the visible error or {log_path}')

run_live([sys.executable,'-m','pip','uninstall','-y','gradio','gradio-client','diffusers'],ROOT/'logs/setup.log')
run_live([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements.txt')],ROOT/'logs/setup.log')
smoke="""
import torch, bitsandbytes as bnb
assert torch.cuda.is_available(), 'Enable T4 GPU'
assert 'T4' in torch.cuda.get_device_name(0), 'Assignment requires T4'
x=torch.randn(128,128,device='cuda',dtype=torch.float16)
q,state=bnb.functional.quantize_4bit(x,quant_type='nf4')
y=bnb.functional.dequantize_4bit(q,state)
torch.cuda.synchronize()
assert y.shape==x.shape and torch.isfinite(y).all()
import pipeline
print('GPU TEST PASSED:',torch.cuda.get_device_name(0),'Torch',torch.__version__,'bitsandbytes',bnb.__version__)
"""
run_live([sys.executable,'-u','-c',smoke],ROOT/'logs/preflight.log')
run_live([sys.executable,'-m','unittest','-v','test_audit'],ROOT/'logs/tests.log')


Log: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/logs/setup.log
Found existing installation: gradio 6.31.0
Uninstalling gradio-6.31.0:
  Successfully uninstalled gradio-6.31.0
Found existing installation: gradio_client 2.7.2
Uninstalling gradio_client-2.7.2:
  Successfully uninstalled gradio_client-2.7.2
Found existing installation: diffusers 0.42.1
Uninstalling diffusers-0.42.1:
  Successfully uninstalled diffusers-0.42.1
Log: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/logs/setup.log
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 92.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 38.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.6 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokeniz

In [4]:
PRIVATE=ROOT/'private_data'
run_live([sys.executable,'-u','audit.py','--train',str(PACK/'data/train.jsonl'),'--out',str(PRIVATE)],ROOT/'logs/audit.log')
print('Inspect proposed corrections:',PRIVATE/'audit_changes.jsonl')
print('Original data is unchanged. No dev/test rows are used by the cleaner.')


Log: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/logs/audit.log
{
  "input_rows": 1445,
  "kept_rows": 1360,
  "dropped_rows": 85,
  "fixed_rows": 218,
  "exact_duplicate_extras": 60,
  "whitespace_normalized_duplicate_extras": 77,
  "long_inputs_over_1000_chars": 40,
  "conflicting_input_groups": 0,
  "reasons": {
    "schema priority precedence": 62,
    "legacy priority key/value": 70,
    "legacy date format": 30,
    "invalid/truncated target JSON; no label reconstruction": 25,
    "normalize transaction ID": 20,
    "unique explicit claim amount": 21,
    "remove quoted earlier emails from training input": 48,
    "exact duplicate of tr-00030": 1,
    "exact duplicate of tr-00150": 1,
    "exact duplicate of tr-00137": 1,
    "exact duplicate of tr-00070": 1,
    "exact duplicate of tr-00142": 1,
    "exact duplicate of tr-00217": 1,
    "exact duplicate of tr-00173": 1,
    "exact duplicate of tr-00345": 1,
    "exact duplicate of tr-00175": 1,
    "exact duplicate of tr

## Run experiments with visible progress
First use `MODE="explore"`. For a disconnection: reconnect, rerun setup cells, and rerun this cell with the same mode/output name. Completed results are reused; saved checkpoints and prediction prefixes are resumed. Do not change data/code/settings while resuming.

After exploration, inspect results. Switch to `reproduce` on a fresh T4 runtime and choose a new REPRO_NAME for the final timing check. The final submitted notebook must use reproduce mode and include outputs. A resumed/reused run cannot certify a fresh under-three-hour execution.

In [5]:
MODE = 'reproduce'
REPRO_NAME = 'reproduction_02'  # Use a new name for a fresh timing check.
OUT=ROOT/'runs'/('exploration' if MODE=='explore' else REPRO_NAME)
BEST=PROJECT/'best_config.json'
if MODE=='reproduce': assert BEST.exists(), 'Finish exploration first'
# Refuse a concurrent copy of this pipeline.
for proc in Path('/proc').iterdir():
    if not proc.name.isdigit(): continue
    try:
        args=(proc/'cmdline').read_bytes().split(b'\0')
        if any(a==b'pipeline.py' or a.decode(errors='ignore').endswith('/pipeline.py') for a in args):
            raise RuntimeError(f'Another pipeline is running (PID {proc.name}); do not launch twice.')
    except (FileNotFoundError,PermissionError,ProcessLookupError): pass
command=[sys.executable,'-u',str(PROJECT/'pipeline.py'),'--mode',MODE,
         '--pack',str(PACK),'--private',str(PRIVATE),'--out',str(OUT),
         '--best-config',str(BEST)]
run_live(command,ROOT/'logs'/f'{MODE}_{time.time_ns()}.log')
print('FINISHED:',MODE)
print((OUT/'runtime.json').read_text())


Log: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/logs/reproduce_1794521975533045102.log
I0000 00:00:1794521983.278309    1888 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794521985.790658    1888 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
STARTING: final
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Training checkpoint: fresh run

  6%|▌         | 10/180 [01:10<18:40,  6.59s/it]
               

In [6]:
print('EXPERIMENT RESULTS')
for path in sorted((ROOT/'runs/exploration').glob('*/metrics.json')):
    m=json.loads(path.read_text())
    print(m['run'],'EM:',f"{m['exact_match']:.2%}",
          'Fields:',f"{m['mean_field_acc']:.2%}",
          'Train minutes:',round(m['training_minutes'],2),m.get('timing_scope'))
print('Failures for review:',OUT/'submission/failures.jsonl')
if MODE=='reproduce':
    DEST=ROOT/f'submission_repo_{REPRO_NAME}'
    run_live([sys.executable,'make_submission.py','--source',str(PROJECT),
              '--explore',str(ROOT/'runs/exploration'),'--final',str(OUT),
              '--dest',str(DEST)],ROOT/'logs/export.log')
    print('Exported:',DEST)
    print('Complete report explanations. Save this executed notebook as notebook.ipynb in that folder.')
else:
    print('Review results before running fresh-runtime reproduction. Do not submit exploration alone.')
print('Elapsed notebook minutes:',round((time.perf_counter()-STARTED)/60,2))


EXPERIMENT RESULTS
clean_fp16 EM: 66.00% Fields: 93.94% Train minutes: 18.62 full_training
clean_qlora EM: 63.50% Fields: 93.38% Train minutes: 26.1 full_training
compact_fp16 EM: 62.50% Fields: 93.25% Train minutes: 18.75 full_training
rank32_fp16 EM: 63.00% Fields: 93.31% Train minutes: 18.83 full_training
raw_fp16 EM: 71.50% Fields: 94.56% Train minutes: 20.42 full_training
Failures for review: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/runs/reproduction_02/submission/failures.jsonl
Log: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/logs/export.log
Exported /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/submission_repo_reproduction_02 — complete report TODOs and save executed notebook before submission.
Exported: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/submission_repo_reproduction_02
Complete report explanations. Save this executed notebook as notebook.ipynb in that folder.
Elapsed notebook minutes: 62.26


## Before submission
Keep the repository private until the confirmed deadline. Exclude original/cleaned training data and base weights. Include the final adapter, raw test predictions, cleaning log, completed ≤3-page report, README and executed reproduction notebook. The exported report contains explicit TODOs requiring real failure explanations; it is not finished automatically. Preserve the AI disclosure.

Checkpoint recovery saves work but resumed timing is only the last successful training segment. Entries marked `resumed_segment_only` must not be reported as complete training time. Fresh reproduction must run top to bottom, including setup, in under three hours; measure it rather than assume it.